In [284]:
#data collection

import nltk


In [285]:
nltk.download('gutenberg')

[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/adityakumar/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


True

In [286]:
import nltk
from nltk.corpus import gutenberg


nltk.download('gutenberg')

#all shakespeare files in gutenberg
shakespeare_files = [fileid for fileid in gutenberg.fileids() if 'shakespeare' in fileid]

# Combined txt download
combined_text = ""
for fileid in shakespeare_files:
    combined_text += gutenberg.raw(fileid) + "\n"


import re

text = re.sub(r'[^\w\s\']', ' ', combined_text)
text=text.lower()

# Save combined txt
output_filename = "shakespeare_lstm_input.txt"
with open(output_filename, "w") as f:
    f.write(text)

print(text[:500])

 the tragedie of julius caesar by william shakespeare 1599 


actus primus  scoena prima 

enter flauius  murellus  and certaine commoners ouer the stage 

  flauius  hence  home you idle creatures  get you home 
is this a holiday  what  know you not
 being mechanicall  you ought not walke
vpon a labouring day  without the signe
of your profession  speake  what trade art thou 
  car  why sir  a carpenter

   mur  where is thy leather apron  and thy rule 
what dost thou with thy best apparrell on


[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/adityakumar/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [ ]:

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])
total_words=len(tokenizer.word_index)+1
print(total_words)

7804


In [288]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'to': 3,
 'i': 4,
 'of': 5,
 'you': 6,
 'a': 7,
 'my': 8,
 'that': 9,
 'in': 10,
 'it': 11,
 'is': 12,
 'not': 13,
 'his': 14,
 'with': 15,
 'this': 16,
 'your': 17,
 'me': 18,
 'for': 19,
 'but': 20,
 'he': 21,
 'be': 22,
 'haue': 23,
 'him': 24,
 'what': 25,
 'as': 26,
 'so': 27,
 'will': 28,
 'our': 29,
 'ham': 30,
 'all': 31,
 'thou': 32,
 'we': 33,
 'are': 34,
 'shall': 35,
 'no': 36,
 'lord': 37,
 'then': 38,
 'on': 39,
 'do': 40,
 'by': 41,
 'if': 42,
 'come': 43,
 'enter': 44,
 'king': 45,
 'they': 46,
 'good': 47,
 'now': 48,
 'thy': 49,
 'caesar': 50,
 'let': 51,
 'from': 52,
 'at': 53,
 'was': 54,
 'which': 55,
 'or': 56,
 'vs': 57,
 'them': 58,
 'did': 59,
 'more': 60,
 'thee': 61,
 'their': 62,
 'know': 63,
 'there': 64,
 'brutus': 65,
 'like': 66,
 'would': 67,
 'when': 68,
 'how': 69,
 'vpon': 70,
 'bru': 71,
 'well': 72,
 'her': 73,
 'hath': 74,
 'selfe': 75,
 'am': 76,
 'man': 77,
 'macb': 78,
 'yet': 79,
 'why': 80,
 'should': 81,
 'may': 82,
 '

In [290]:
encoded = tokenizer.texts_to_sequences([text])[0]

input_seq_len = 30
sequences = []

for i in range(input_seq_len, len(encoded)):
    sequences.append(encoded[i-input_seq_len:i+1])

input_sequence = np.array(sequences)

In [291]:
input_sequence

array([[   1,  837,    5, ...,  300,    6,  363],
       [ 837,    5, 3552, ...,    6,  363,   12],
       [   5, 3552,   50, ...,  363,   12,   16],
       ...,
       [1463,   10, 2003, ..., 2124,    1,  837],
       [  10, 2003,   94, ...,    1,  837,    5],
       [2003,   94,    2, ...,  837,    5,  149]], shape=(68031, 31))

In [292]:
print ('Total Sequences: %d' % len(input_sequence))
print ('This is the first sequence: {0}'.format(input_sequence[0]))

Total Sequences: 68031
This is the first sequence: [   1  837    5 3552   50   41 1816 1817 2389  616 1818 1819  935   44
 1233 2390    2  520 2391  379    1  838 1233  305  363    6  936 1234
  300    6  363]


In [296]:
#create predictors and labels
import tensorflow as tf
x,y=input_sequence[:,:-1],input_sequence[:,-1]

In [297]:
len(input_sequence)

68031

In [298]:
#train test split
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=42)

In [299]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping=EarlyStopping(monitor='val_loss',patience=7,restore_best_weights=True)

In [ ]:
'''
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.37,      
    patience=10,        
    min_lr=0.00001,   
    verbose=1
)'''

In [ ]:
#parameters

#input_seq_len
dimension_to_represent_word=100
#total_words = len(tokenizer.word_index) + 1

In [302]:
#train lstm 

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,Input
import tensorflow as tf



#define the model
model=Sequential([
Input(shape=(input_seq_len,)),
Embedding(total_words,dimension_to_represent_word,mask_zero=True),
Dropout(0.1),
LSTM(150,return_sequences=True,recurrent_dropout=0.1),
Dropout(0.3),
LSTM(100),
Dense(total_words,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

print(model.summary())


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 30, 100)        │       780,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 30, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 30, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 30, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7804)           │       788,204 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,819,604 (6.94 MB)

 Trainable params: 1,819,604 (6.94 MB)

 Non-trainable params: 0 (0.00 B)

None


In [303]:
#train model
history=model.fit(X_train,y_train,epochs=100,validation_data=(X_test,y_test),verbose=1,callbacks=[early_stopping])

Epoch 1/100
1595/1595 ━━━━━━━━━━━━━━━━━━━━ 92s 56ms/step - accuracy: 0.0327 - loss: 6.9334 - val_accuracy: 0.0355 - val_loss: 6.8050
Epoch 2/100
1595/1595 ━━━━━━━━━━━━━━━━━━━━ 106s 66ms/step - accuracy: 0.0424 - loss: 6.5438 - val_accuracy: 0.0433 - val_loss: 6.8024
Epoch 3/100
1595/1595 ━━━━━━━━━━━━━━━━━━━━ 106s 67ms/step - accuracy: 0.0471 - loss: 6.3576 - val_accuracy: 0.0523 - val_loss: 6.7858
Epoch 4/100
1595/1595 ━━━━━━━━━━━━━━━━━━━━ 97s 61ms/step - accuracy: 0.0586 - loss: 6.1742 - val_accuracy: 0.0651 - val_loss: 6.7886
Epoch 5/100
1595/1595 ━━━━━━━━━━━━━━━━━━━━ 104s 65ms/step - accuracy: 0.0728 - loss: 5.9824 - val_accuracy: 0.0693 - val_loss: 6.8452
Epoch 6/100
1595/1595 ━━━━━━━━━━━━━━━━━━━━ 106s 66ms/step - accuracy: 0.0811 - loss: 5.8165 - val_accuracy: 0.0736 - val_loss: 6.8784
Epoch 7/100
1595/1595 ━━━━━━━━━━━━━━━━━━━━ 135s 85ms/step - accuracy: 0.0859 - loss: 5.6579 - val_accuracy: 0.0734 - val_loss: 6.9772
Epoch 8/100
1595/1595 ━━━━━━━━━━━━━━━━━━━━ 102s 64ms/step - accu

In [304]:
def prediction(
    text: str,
    n_word: int,
    k: int = 7,
    temperature: float = 0.9
) -> str:
    """
    Generate n_word next words using temperature-scaled top-k sampling.

    Args:
        text: Seed text.
        n_word: Number of words to generate.
        k: Sample only from top-k predictions.
        temperature:
            <1.0 = more conservative
            1.0 = unchanged
            >1.0 = more random
    """

    for _ in range(n_word):

        # Convert text to token ids
        token_text = tokenizer.texts_to_sequences([text])[0]

        # Pad to model input length
        padded_token_input = pad_sequences(
            [token_text],
            maxlen=input_seq_len,
            padding="pre",
            truncating="pre"
        )

        # Predict probabilities
        probs = model.predict(
            padded_token_input,
            verbose=0
        )[0]

        # Temperature scaling
        probs = np.log(probs + 1e-10) / temperature
        probs = np.exp(probs)
        probs = probs / probs.sum()

        # Top-k selection
        top_k_indices = np.argsort(probs)[-k:]

        top_k_probs = probs[top_k_indices]
        top_k_probs = top_k_probs / top_k_probs.sum()

        # Sample
        predicted_index = np.random.choice(
            top_k_indices,
            p=top_k_probs
        )

        # Index -> word
        predicted_word = tokenizer.index_word.get(
            predicted_index,
            "<UNK>"
        )

        # Append word
        text += " " + predicted_word

    return text

In [317]:
text="i am going to meet "
n_word=5
prediction(text,n_word)

'i am going to meet  to the heart of the'

In [306]:
model.save('shekspeare2.h5')

In [307]:
import pickle
with open('tokenzer.pkl','wb') as handle:
    pickle.dump(tokenizer,handle,protocol=pickle.HIGHEST_PROTOCOL)